# Model Comparison Notebook

Evaluates multiple LLM configs (base model vs LoRA-adapted) across all 38 trading symbols over 2022–2025.
For each model × symbol, generates `N_SAMPLES` strategy candidates and scores them through the full reward pipeline.
Outputs per-model stats, top strategies per symbol, and cross-model comparison charts.

### Installations

In [ ]:
%%capture
import os, re
if 'COLAB_' not in ''.join(os.environ.keys()):
    %pip install unsloth
else:
    import torch; v = re.match(r'[0-9]{1,}\.[0-9]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + ('0.0.33.post1' if v=='2.9' else '0.0.32.post2' if v=='2.8' else '0.0.29.post3')
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth

In [ ]:
%pip install backtrader alpaca_trade_api plotly pandas -q

### Secrets

In [ ]:
HF_TOKEN=''
ALPACA_API_KEY=''
ALPACA_SECRET_KEY=''

### Configuration

Edit `MODEL_CONFIGS` to add/remove models. Set `lora_adapter_path` to a HuggingFace repo ID or local path, or `None` for the base model.

In [ ]:
MODEL_CONFIGS = [
    {
        'label': 'Qwen2.5-32B base',
        'model_name': 'unsloth/Qwen2.5-Coder-32B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Qwen2.5-32B LoRA v1-500',
        'model_name': 'unsloth/Qwen2.5-Coder-32B-Instruct',
        'lora_adapter_path': 'adhamhelmy/qwen2.5-coder-32b-instruct-v1-500',
        'base_label': 'Qwen2.5-32B base',
    },
    {
        'label': 'Llama-3.1-8B base',
        'model_name': 'unsloth/Meta-Llama-3.1-8B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Llama-3.1-8B LoRA v1-500',
        'model_name': 'unsloth/Meta-Llama-3.1-8B-Instruct',
        'lora_adapter_path': 'adhamhelmy/meta-llama-3.1-8b-instruct-v1-500',
        'base_label': 'Llama-3.1-8B base',
    },
    {
        'label': 'Qwen2.5-7B base',
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct',
        'lora_adapter_path': None,
    },
    {
        'label': 'Qwen2.5-7B LoRA v1-500',
        'model_name': 'unsloth/Qwen2.5-Coder-7B-Instruct',
        'lora_adapter_path': 'adhamhelmy/qwen2.5-coder-7b-instruct-v1-500',
        'base_label': 'Qwen2.5-7B base',
    },
]

TEST_START = '2022-01-01'
TEST_END   = '2025-12-31'
N_SAMPLES  = 5   # strategy candidates per symbol per model

SYMBOLS = [
    'AAPL', 'AMGN', 'AXP',  'BA',   'CAT',
    'CRM',  'CSCO', 'CVX',  'DIS',  'DOW',
    'GS',   'HD',   'HON',  'IBM',  'JNJ',
    'JPM',  'KO',   'MCD',  'MMM',  'MRK',
    'MSFT', 'NKE',  'NVDA', 'PG',   'TRV',
    'UNH',  'V',    'VZ',   'WBA',  'WMT',
    'AMZN', 'COIN', 'GE',   'GOOGL','NFLX',
    'NIO',  'TSLA', 'UVV',
]

### Core Classes

In [ ]:
import torch
import re
import os
import json
import shutil
from datetime import datetime

import backtrader as bt
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

from peft import PeftModel
from unsloth import FastVisionModel, execute_with_time_limit, check_python_modules
from transformers import TextStreamer
from alpaca_trade_api.rest import REST, TimeFrame
from IPython.display import Image, display

In [ ]:
class Unsloth:
    def __init__(self, model_name, lora_rank=32, max_seq_length=4096,
                 load_in_4bit=True, fast_inference=False, max_prompt_length=512,
                 lora_adapter_path=None):
        self.model_name = model_name
        self.lora_rank = lora_rank
        self.max_seq_length = max_seq_length
        self.max_prompt_length = max_prompt_length

        self.model, self.tokenizer = FastVisionModel.from_pretrained(
            model_name=model_name,
            max_seq_length=max_seq_length,
            load_in_4bit=load_in_4bit,
            fast_inference=fast_inference,
        )

        if lora_adapter_path:
            self.model = PeftModel.from_pretrained(
                self.model,
                lora_adapter_path,
                token=HF_TOKEN,
                is_trainable=False,
            )
        else:
            self.model = FastVisionModel.get_peft_model(
                self.model,
                r=lora_rank,
                target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                'gate_proj', 'up_proj', 'down_proj'],
                lora_alpha=lora_rank * 2,
                use_gradient_checkpointing='unsloth',
                random_state=3407,
            )

    def generate(self, inputs):
        text = self.tokenizer.apply_chat_template(
            [{'role': 'user', 'content': inputs.strip()}],
            tokenize=False,
            add_generation_prompt=True,
        )
        output = self.model.generate(
            **self.tokenizer(images=None, text=text, return_tensors='pt').to('cuda'),
            temperature=1.0,
            do_sample=True,
            max_new_tokens=self.max_seq_length - self.max_prompt_length,
        )
        return self.tokenizer.decode(output[0], skip_special_tokens=True)

    def unload(self):
        del self.model, self.tokenizer
        torch.cuda.empty_cache()
        print('Model unloaded and GPU cache cleared.')

In [ ]:
class Backtrader:
    """Singleton with data cache shared across model evaluations."""
    _instance = None
    _data_cache = {}

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if hasattr(self, '_initialized'):
            return
        self._initialized = True
        self.rest_api = REST(ALPACA_API_KEY, ALPACA_SECRET_KEY, 'https://paper-api.alpaca.markets')

    def _get_bars(self, symbol, timeframe, start, end):
        key = (symbol, str(timeframe), start, end)
        if key not in Backtrader._data_cache:
            Backtrader._data_cache[key] = self.rest_api.get_bars(
                symbol, timeframe, start, end, adjustment='all'
            ).df
        return Backtrader._data_cache[key]

    def load_bars(self, symbols, start, end, timeframe=TimeFrame.Day):
        print(f'Pre-fetching {len(symbols)} symbols ({start} to {end})...')
        for i, sym in enumerate(symbols, 1):
            self._get_bars(sym, timeframe, start, end)
            print(f'  [{i}/{len(symbols)}] {sym}')
        print(f'Done. {len(Backtrader._data_cache)} bar series cached.')

    def run_backtest(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10000, plot=False):
        cerebro = bt.Cerebro(stdstats=True)
        cerebro.broker.setcash(cash)
        cerebro.addstrategy(strategy)
        cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='mysharpe')
        cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='annual_return')
        cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

        if isinstance(symbols, str):
            cerebro.adddata(bt.feeds.PandasData(
                dataname=self._get_bars(symbols, timeframe, start, end), name=symbols
            ))
        else:
            for sym in symbols:
                cerebro.adddata(bt.feeds.PandasData(
                    dataname=self._get_bars(sym, timeframe, start, end), name=sym
                ))

        init_val = cerebro.broker.getvalue()
        results = cerebro.run()
        _return = (cerebro.broker.getvalue() / init_val - 1) * 100

        strat = results[0]
        sharpe = strat.analyzers.mysharpe.get_analysis().get('sharperatio')
        annual = strat.analyzers.annual_return.get_analysis()
        avg_annual = (sum(annual.values()) / len(annual) * 100) if annual else 0.0
        max_dd = strat.analyzers.drawdown.get_analysis()['max']['drawdown']

        if plot and _return > 0:
            cerebro.plot(iplot=False)
            for i, fig_num in enumerate(plt.get_fignums(), start=1):
                plt.figure(fig_num)
                fname = f'backtest_plot_{i}.png'
                plt.savefig(fname, dpi=140, bbox_inches='tight')
                display(Image(fname))
            plt.close('all')

        return _return, sharpe, avg_annual, max_dd

    @execute_with_time_limit(10)
    def timed_backtest(self, strategy, symbols, start, end, timeframe=TimeFrame.Day, cash=10000):
        return self.run_backtest(strategy, symbols, start, end, timeframe, cash, plot=False)

In [ ]:
class RewardFunctions:

    def extract_function(text):
        if text.count('```') >= 2:
            first = text.find('```') + 3
            second = text.find('```', first)
            fx = text[first:second].strip().removeprefix('python\n')
            fx = fx[fx.find('class Strategy'):]
            if fx.startswith('class Strategy(bt.Strategy):'):
                return fx
        idx = text.find('class Strategy(bt.Strategy):')
        if idx != -1:
            return text[idx:]
        return None

    def function_works(function):
        if function is None:
            return False
        ok, info = check_python_modules(function)
        return not (ok is False or 'error' in info)

    def has_required_functions(text):
        has_init = bool(re.search(r'def\s+__init__\s*\([^)]*\)\s*:', text))
        has_next = bool(re.search(r'def\s+next\s*\([^)]*\)\s*:', text))
        return has_init and has_next

    def extract_strategy(func):
        namespace = {'bt': bt}
        exec(func, namespace)
        return namespace['Strategy']

### Evaluation Functions

In [ ]:
def make_prompt(symbol, start, end):
    lines = [
        f'Create a trading strategy for {symbol} from {start} to {end} that is fully compatible with the following backtesting setup:',
        '',
        '- Framework: Backtrader',
        '- Strategy must subclass bt.Strategy',
        '- The strategy will be passed directly into:',
        'run_backtest(StrategyClass, symbols, start, end, timeframe, cash)',
        '',
        'STRICT RULES:',
        '1. Output ONLY a single Python class definition (no explanations, no markdown, no comments outside the class).',
        '2. The class MUST be named Strategy.',
        '3. Do NOT include imports (bt is already available).',
        '4. Do NOT reference external data, files, APIs, or indicators outside Backtrader.',
        '5. The strategy MUST work for single-symbol backtests.',
        '6. All indicators must be created in __init__.',
        '7. Trading logic must be implemented in next().',
        '8. Orders must use only: self.buy(), self.sell(), self.close(), self.order_target_percent().',
        '9. No plotting, printing, logging, or analyzers.',
        '10. Strategy must be deterministic and backtest-safe (no lookahead bias).',
        '',
        'Return ONLY the Python class. DO NOT output anything else.',
    ]
    return '\n'.join(lines)

In [ ]:
def evaluate_symbol(model, bt_instance, symbol, start, end, n_samples):
    """Generate and score n_samples strategy candidates for one symbol."""
    prompt = make_prompt(symbol, start, end)
    results = []

    for i in range(n_samples):
        rec = {
            'symbol': symbol, 'sample': i + 1,
            'status': None, 'reward_score': None,
            'return_pct': None, 'sharpe_ratio': None,
            'avg_annual_return_pct': None, 'max_drawdown_pct': None,
            'strategy_code': None,
        }

        try:
            raw = model.generate(prompt)
            func = RewardFunctions.extract_function(raw)
            rec['strategy_code'] = func

            if not RewardFunctions.has_required_functions(func or ''):
                rec.update({'status': 'missing_methods', 'reward_score': -10})
                results.append(rec)
                continue

            if not RewardFunctions.function_works(func):
                rec.update({'status': 'invalid_code', 'reward_score': -3})
                results.append(rec)
                continue

            strategy_cls = RewardFunctions.extract_strategy(func)
            _ret, sharpe, avg_ann, max_dd = bt_instance.timed_backtest(
                strategy_cls, symbol, start, end
            )
            rec['return_pct'] = _ret
            rec['sharpe_ratio'] = sharpe
            rec['avg_annual_return_pct'] = avg_ann
            rec['max_drawdown_pct'] = max_dd

            if _ret == 0 and sharpe is None:
                rec.update({'status': 'no_trades', 'reward_score': -1})
            elif _ret > 0:
                rec.update({'status': 'profitable', 'reward_score': max(avg_ann, 1)})
            else:
                rec.update({'status': 'loss', 'reward_score': 0})

        except TimeoutError:
            rec.update({'status': 'timeout', 'reward_score': -2})
        except Exception as e:
            rec.update({'status': 'exception', 'reward_score': -2})
            print(f'    [{symbol} s{i+1}] Exception: {str(e)[:100]}')

        results.append(rec)

    scores = [r['reward_score'] for r in results if r['reward_score'] is not None]
    statuses = [r['status'] for r in results]
    print(f'  {symbol}: {statuses}  scores={scores}')
    return results

In [ ]:
def evaluate_model(config, bt_instance, symbols=None, start=TEST_START, end=TEST_END, n_samples=N_SAMPLES):
    """Load model, evaluate all symbols, unload model, return flat list of result records."""
    symbols = symbols or SYMBOLS
    label = config['label']
    print(f"\n{'='*60}")
    print(f'Evaluating: {label}')
    print(f'Model: {config["model_name"]}')
    adapter = config.get('lora_adapter_path')
    print(f'LoRA adapter: {adapter or "None (base model)"}')
    print(f'Symbols: {len(symbols)}  |  Samples/symbol: {n_samples}  |  Period: {start} to {end}')
    print(f"{'='*60}")

    model = Unsloth(
        model_name=config['model_name'],
        max_seq_length=1024,
        lora_adapter_path=adapter,
    )

    records = []
    for sym in symbols:
        sym_recs = evaluate_symbol(model, bt_instance, sym, start, end, n_samples)
        for r in sym_recs:
            r['model'] = label
        records.extend(sym_recs)

    model.unload()

    ok = sum(1 for r in records if r['status'] == 'profitable')
    print(f'\n[{label}] Done. Profitable: {ok}/{len(records)}')
    return records

### Run Evaluation

In [ ]:
# Initialize Backtrader singleton once — data cache persists across model switches
bt_instance = Backtrader()
bt_instance.load_bars(SYMBOLS, TEST_START, TEST_END)

In [ ]:
all_results = []
for config in MODEL_CONFIGS:
    records = evaluate_model(config, bt_instance)
    all_results.extend(records)

df = pd.DataFrame(all_results)
print(f'\nTotal samples: {len(df)}')
df.head(10)

In [ ]:
df.to_csv('model_comparison_results.csv', index=False)
print('Saved to model_comparison_results.csv')

### Results Analysis

In [ ]:
# Per-model summary statistics
summary = df.groupby('model').agg(
    total=('sample', 'count'),
    profitable=('status', lambda x: (x == 'profitable').sum()),
    loss=('status', lambda x: (x == 'loss').sum()),
    no_trades=('status', lambda x: (x == 'no_trades').sum()),
    invalid=('status', lambda x: x.isin(['missing_methods', 'invalid_code', 'exception', 'timeout']).sum()),
    mean_reward=('reward_score', 'mean'),
    median_reward=('reward_score', 'median'),
    mean_return=('return_pct', 'mean'),
    mean_sharpe=('sharpe_ratio', 'mean'),
    mean_annual=('avg_annual_return_pct', 'mean'),
).reset_index()
summary['profitable_pct'] = (summary['profitable'] / summary['total'] * 100).round(1)
display(summary.round(3))

In [ ]:
# Reward score distribution by model
fig = go.Figure()
for label in df['model'].unique():
    scores = df[df['model'] == label]['reward_score'].dropna()
    fig.add_trace(go.Box(
        y=scores, name=label,
        boxpoints='all', jitter=0.3, pointpos=-1.5,
    ))
fig.update_layout(
    title='Reward Score Distribution by Model',
    yaxis_title='Reward Score',
    template='plotly_dark',
)
fig.show()

In [ ]:
# Outcome breakdown (stacked bar)
status_counts = df.groupby(['model', 'status']).size().reset_index(name='count')
fig = px.bar(
    status_counts, x='model', y='count', color='status', barmode='stack',
    title='Outcome Breakdown by Model',
    color_discrete_map={
        'profitable': '#2ecc71', 'loss': '#e67e22', 'no_trades': '#95a5a6',
        'missing_methods': '#e74c3c', 'invalid_code': '#c0392b',
        'exception': '#9b59b6', 'timeout': '#7f8c8d',
    },
    template='plotly_dark',
)
fig.show()

In [ ]:
# Best reward per model per symbol (highest scoring sample)
best = (
    df[df['reward_score'].notna()]
    .sort_values('reward_score', ascending=False)
    .groupby(['model', 'symbol'])
    .first()
    .reset_index()
)

# Cross-model comparison: best reward per symbol
pivot = best.pivot_table(index='symbol', columns='model', values='reward_score')
melted = pivot.reset_index().melt(id_vars='symbol', var_name='model', value_name='best_reward')
fig = px.bar(
    melted, x='symbol', y='best_reward', color='model', barmode='group',
    title='Best Reward Score per Symbol — Model Comparison',
    template='plotly_dark',
)
fig.update_layout(xaxis_tickangle=45, height=500)
fig.show()

In [ ]:
# Return % heatmap per model
for label in df['model'].unique():
    sub = best[best['model'] == label][['symbol', 'return_pct']].set_index('symbol')
    fig = px.imshow(
        sub.T,
        title=f'Best Return % per Symbol — {label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Mean reward per model per symbol (heatmap)
for label in df['model'].unique():
    sub = df[df['model'] == label].groupby('symbol')['reward_score'].mean().reset_index()
    sub.columns = ['symbol', 'mean_reward']
    sub = sub.set_index('symbol')
    fig = px.imshow(
        sub.T,
        title=f'Mean Reward Score per Symbol — {label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.2f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Return delta: each LoRA vs its paired base model
pivot_ret = best.pivot_table(index='symbol', columns='model', values='return_pct')

for config in MODEL_CONFIGS:
    if not config.get('lora_adapter_path') or not config.get('base_label'):
        continue
    adapted_label = config['label']
    base_label    = config['base_label']
    if base_label not in pivot_ret.columns or adapted_label not in pivot_ret.columns:
        continue
    delta = (pivot_ret[adapted_label] - pivot_ret[base_label]).to_frame(name='delta_return_pct')
    fig = px.imshow(
        delta.T,
        title=f'Return Delta: {adapted_label} vs {base_label}',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0,
        text_auto='.1f',
        template='plotly_dark',
    )
    fig.update_layout(height=200, xaxis_tickangle=45)
    fig.show()

In [ ]:
# Save best profitable strategy per model per symbol
save_dir = 'model_comparison_strategies'
os.makedirs(save_dir, exist_ok=True)
saved = 0

for _, row in best[best['status'] == 'profitable'].iterrows():
    model_dir = os.path.join(save_dir, row['model'].replace('/', '_').replace(' ', '_'))
    os.makedirs(model_dir, exist_ok=True)

    code = row.get('strategy_code')
    if code:
        with open(os.path.join(model_dir, f"{row['symbol']}_strategy.py"), 'w') as f:
            f.write(code)

    stats = {
        'model': row['model'],
        'symbol': row['symbol'],
        'return_pct': float(row['return_pct']) if pd.notna(row['return_pct']) else None,
        'sharpe_ratio': float(row['sharpe_ratio']) if pd.notna(row['sharpe_ratio']) else None,
        'avg_annual_return_pct': float(row['avg_annual_return_pct']) if pd.notna(row['avg_annual_return_pct']) else None,
        'max_drawdown_pct': float(row['max_drawdown_pct']) if pd.notna(row['max_drawdown_pct']) else None,
    }
    with open(os.path.join(model_dir, f"{row['symbol']}_stats.json"), 'w') as fj:
        json.dump(stats, fj, indent=2)
    saved += 1

print(f'Saved {saved} profitable strategies to {save_dir}/')